In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ELAN_MASTER = Path("elan_master.csv")
OUT_DIR     = Path("window")

WIN = 4.0
HOP = 1.0

TRAIN_FRAC = 0.8
VAL_FRAC   = 0.1

BG_RATIO = 2.0   
SEED = 42

LABEL_MAP = {
    "MF": ["mindfulness"],
    "SK": ["self_kindness", "common_humanity"],
    "SJ": ["self_judgement", "over_identified", "isolation"],
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ELAN_MASTER:", ELAN_MASTER)
print("OUT_DIR:", OUT_DIR)


In [ ]:
def overlap_fraction(a_start: float, a_end: float, b_start: float, b_end: float) -> float:
    inter = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    denom = (b_end - b_start)
    return 0.0 if denom <= 0 else inter / denom

def build_windows(t_max: float, win: float, hop: float) -> pd.DataFrame:
    if t_max <= win:
        starts = np.array([0.0], dtype=float)
    else:
        starts = np.arange(0.0, (t_max - win) + 1e-6, hop, dtype=float)
    return pd.DataFrame({
        "w_start": starts,
        "w_end": starts + win,
        "w_center": starts + win / 2.0,
    })

def label_windows_for_video(df_vid: pd.DataFrame, video_id: str, win: float, hop: float, label_map: dict) -> pd.DataFrame:
    t_max = float(df_vid["t_end"].max())
    windows = build_windows(t_max, win=win, hop=hop)

    out_rows = []
    for w in windows.itertuples(index=False):
        row = {
            "video_id": video_id,
            "w_start": float(w.w_start),
            "w_end": float(w.w_end),
            "w_center": float(w.w_center),
        }

        any_overlap = False
        for lab, tiers in label_map.items():
            df_lab = df_vid[df_vid["tier"].isin(tiers)]
            y = 0.0
            if not df_lab.empty:
                for r in df_lab.itertuples(index=False):
                    y = max(y, overlap_fraction(float(r.t_start), float(r.t_end), row["w_start"], row["w_end"]))

            mask = 1 if y > 0 else 0
            row[f"y_{lab}"] = float(y)
            row[f"mask_{lab}"] = int(mask)
            if mask == 1:
                any_overlap = True

        row["mask_any"] = int(any_overlap)
        out_rows.append(row)

    return pd.DataFrame(out_rows)


In [ ]:
def make_video_strat_table(elan_filtered: pd.DataFrame, label_map: dict) -> pd.DataFrame:
    vids = []
    for video_id, df_vid in elan_filtered.groupby("video_id"):
        feats = {}
        for lab, tiers in label_map.items():
            df_lab = df_vid[df_vid["tier"].isin(tiers)]
            dur = float((df_lab["t_end"] - df_lab["t_start"]).clip(lower=0).sum()) if not df_lab.empty else 0.0
            feats[lab] = dur

        def bin_minutes(seconds: float) -> str:
            m = seconds / 60.0
            if m <= 0:  return "0"
            if m <= 2:  return "0-2"
            if m <= 5:  return "2-5"
            if m <= 10: return "5-10"
            return "10+"

        key = f"MF:{bin_minutes(feats['MF'])}|SK:{bin_minutes(feats['SK'])}|SJ:{bin_minutes(feats['SJ'])}"
        vids.append({"video_id": video_id, "strat_key": key})

    return pd.DataFrame(vids)

def split_videos(video_table: pd.DataFrame, train_frac: float, val_frac: float, seed: int) -> dict:
    rng = np.random.default_rng(seed)

    key_counts = video_table["strat_key"].value_counts()
    too_small = int((key_counts < 3).sum())
    use_strat = (too_small == 0) and (len(key_counts) > 1)

    vids_arr = video_table["video_id"].to_numpy()

    if not use_strat:
        rng.shuffle(vids_arr)
        n = len(vids_arr)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        train_ids = vids_arr[:n_train].tolist()
        val_ids = vids_arr[n_train:n_train+n_val].tolist()
        test_ids = vids_arr[n_train+n_val:].tolist()
        return {"train": train_ids, "val": val_ids, "test": test_ids, "stratified": False}

    train_ids, val_ids, test_ids = [], [], []
    df = video_table.copy()
    for _, g in df.groupby("strat_key"):
        g_ids = g["video_id"].tolist()
        rng.shuffle(g_ids)
        n = len(g_ids)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        if n >= 3:
            n_train = max(1, n_train)
            n_val = max(1, n_val)
            if n_train + n_val >= n:
                n_val = max(1, n - n_train - 1)

        train_ids.extend(g_ids[:n_train])
        val_ids.extend(g_ids[n_train:n_train+n_val])
        test_ids.extend(g_ids[n_train+n_val:])

    rng.shuffle(train_ids); rng.shuffle(val_ids); rng.shuffle(test_ids)
    return {"train": train_ids, "val": val_ids, "test": test_ids, "stratified": True}



elan = pd.read_csv(ELAN_MASTER)


all_tiers = sum(LABEL_MAP.values(), [])
elan_f = elan[elan["tier"].isin(all_tiers)].copy()
if elan_f.empty:
    raise RuntimeError("empty")

video_strat = make_video_strat_table(elan_f, LABEL_MAP)
video_strat.to_csv(OUT_DIR / "_video_strat_table.csv", index=False)

splits = split_videos(video_strat, TRAIN_FRAC, VAL_FRAC, SEED)

splits_freeze = {
    "train_videos": splits["train"],
    "val_videos": splits["val"],
    "test_videos": splits["test"],
    "seed": SEED,
    "train_frac": TRAIN_FRAC,
    "val_frac": VAL_FRAC,
    "test_frac": max(0.0, 1.0 - TRAIN_FRAC - VAL_FRAC),
    "window_sec": WIN,
    "hop_sec": HOP,
    "bg_ratio": BG_RATIO,
    "label_map": LABEL_MAP,
    "stratified": splits["stratified"],
    "strat_key_definition": "binned labeled duration per class",
}

with open(OUT_DIR / "splits_freeze.json", "w", encoding="utf-8") as f:
    json.dump(splits_freeze, f, indent=2)


In [ ]:
all_windows = []
for video_id, df_vid in elan_f.groupby("video_id"):
    wdf = label_windows_for_video(df_vid, video_id, WIN, HOP, LABEL_MAP)
    all_windows.append(wdf)

windows_all = pd.concat(all_windows, ignore_index=True)
windows_all.to_csv(OUT_DIR / "windows_all.csv", index=False)

In [ ]:
def sample_background(windows_df: pd.DataFrame, bg_ratio: float, seed: int) -> pd.DataFrame:
    labeled = windows_df[windows_df["mask_any"] == 1].copy()
    bg_pool = windows_df[windows_df["mask_any"] == 0].copy()

    n_bg = int(len(labeled) * bg_ratio)
    n_bg = min(n_bg, len(bg_pool))

    bg = bg_pool.sample(n=n_bg, random_state=seed).copy()

    for lab in ["MF", "SK", "SJ"]:
        bg[f"y_{lab}"] = 0.0
        bg[f"mask_{lab}"] = 1

    bg["is_bg_negative"] = 1
    labeled["is_bg_negative"] = 0

    out = pd.concat([labeled, bg], ignore_index=True)
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


split_map = {vid: "train" for vid in splits_freeze["train_videos"]}
split_map.update({vid: "val" for vid in splits_freeze["val_videos"]})
split_map.update({vid: "test" for vid in splits_freeze["test_videos"]})

windows_all["split"] = windows_all["video_id"].map(split_map)
if windows_all["split"].isna().any():
    bad = windows_all.loc[windows_all["split"].isna(), "video_id"].unique().tolist()
    raise RuntimeError(f"Some video_ids in windows_all not found in split map: {bad[:10]} ...")


for split_name in ["train", "val", "test"]:
    df_split = windows_all[windows_all["split"] == split_name].copy()
    df_sampled = sample_background(df_split, BG_RATIO, SEED)
    df_sampled["split"] = split_name

    out_path = OUT_DIR / f"windows_{split_name}.csv"
    df_sampled.to_csv(out_path, index=False)